In [10]:
# Inference Pipeline: Unwetterwarnung mit Open-Meteo API

In [11]:
# Hopsworks Feature Group und Feature View laden
import os
from pathlib import Path

import hopsworks
from dotenv import load_dotenv

project_root = Path.cwd().resolve()
if not (project_root / ".env").exists():
    project_root = project_root.parent
load_dotenv(project_root / ".env")

api_key = os.getenv("HOPSWORKS_API_KEY")
project_name = os.getenv("HOPSWORKS_PROJECT_NAME")
if not api_key or not project_name:
    raise ValueError("HOPSWORKS_API_KEY und HOPSWORKS_PROJECT_NAME müssen gesetzt sein.")

project = hopsworks.login(
    api_key_value=api_key,
    project=project_name,
    host="eu-west.cloud.hopsworks.ai",
    port=443,
)
fs = project.get_feature_store()
weather_fg = fs.get_feature_group(
    name="weather_features_batch",
    version=1,
)
if weather_fg is None:
    raise RuntimeError("Die Feature Group weather_features_batch wurde nicht gefunden.")

feature_view = fs.get_feature_view(
    name="severe_weather_fv",
    version=1,
)
if feature_view is None:
    raise RuntimeError("Die Feature View severe_weather_fv wurde nicht gefunden.")

latest_batch_features = feature_view.get_batch_data(
    dataframe_type="pandas",
    primary_key=True,
)
if latest_batch_features.empty:
    raise RuntimeError("Die Feature View enthält keine gespeicherten Batch-Features.")

print(f"✅ Feature Group geladen: {weather_fg.name} (v{weather_fg.version})")
print(f"✅ Feature View geladen: {feature_view.name} (v{feature_view.version})")
print(f"📊 Gespeicherte Batch-Features geladen: {len(latest_batch_features)} Zeilen")

2026-09-23 19:54:20,034 INFO: Closing external client and cleaning up certificates.
2026-09-23 19:54:20,039 INFO: Connection closed.
2026-09-23 19:54:20,041 INFO: Initializing external client
2026-09-23 19:54:20,042 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-09-23 19:54:21,163 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/44167
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.12s) 
✅ Feature Group geladen: weather_features_batch (v1)
✅ Feature View geladen: severe_weather_fv (v1)
📊 Gespeicherte Batch-Features geladen: 1956 Zeilen


In [12]:
# Live-Daten abrufen (Forecast von Open-Meteo)
import requests
import pandas as pd
from datetime import datetime, timedelta


def clean_weather_data(df: pd.DataFrame) -> pd.DataFrame:
    """Bereinigt Live-Daten mit denselben Regeln wie die Feature-Pipeline."""
    df = df.copy()
    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    df["location_id"] = df.apply(
        lambda row: f"{row['latitude']}_{row['longitude']}", axis=1
    )
    df = df.dropna(subset=["time", "location_id"])
    df = df.drop_duplicates(subset=["location_id", "time"], keep="last")

    valid_ranges = {
        "temperature_2m": (-90, 60),
        "relative_humidity_2m": (0, 100),
        "precipitation": (0, 1000),
        "pressure_msl": (850, 1100),
        "surface_pressure": (850, 1100),
        "cloud_cover": (0, 100),
        "wind_speed_10m": (0, 250),
        "wind_gusts_10m": (0, 350),
        "cape": (0, 10000),
    }
    numeric_columns = list(valid_ranges)
    for column, (lower, upper) in valid_ranges.items():
        values = pd.to_numeric(df[column], errors="coerce")
        df[column] = values.mask((values < lower) | (values > upper))

    df = df.sort_values(["location_id", "time"])
    for column in numeric_columns:
        df[column] = df.groupby("location_id")[column].transform(
            lambda values: values.interpolate(limit_direction="both")
        )
        df[column] = df[column].fillna(
            df.groupby("location_id")[column].transform("median")
        )

    required_columns = [
        "temperature_2m", "relative_humidity_2m", "precipitation",
        "pressure_msl", "surface_pressure", "cloud_cover",
        "wind_speed_10m", "wind_gusts_10m", "cape",
    ]
    return df.dropna(subset=required_columns).reset_index(drop=True)


def fetch_live_forecast(lat: float, lon: float, location_name: str) -> pd.DataFrame:
    """Holt aktuelle Wetterdaten und einen 3-Tage-Forecast von Open-Meteo."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": [
            "temperature_2m", "relative_humidity_2m", "precipitation", "rain",
            "pressure_msl", "surface_pressure", "cloud_cover",
            "wind_speed_10m", "wind_gusts_10m", "wind_direction_10m", "cape"
        ],
        "past_days": 1,
        "forecast_days": 3,
        "timezone": "UTC"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()["hourly"]

    df = pd.DataFrame(data)
    df["time"] = pd.to_datetime(df["time"])
    df["location"] = location_name
    df["latitude"] = lat
    df["longitude"] = lon
    df = clean_weather_data(df)

    print(f"✅ {len(df)} bereinigte Live-Datenpunkte für {location_name} geladen")
    return df

# Definierte Standorte
LOCATIONS = {
    "Munich": (48.1351, 11.5820),
    "Hamburg": (53.5511, 9.9937)
}

In [13]:
# User-Input für Ad-hoc-Abfrage
def get_user_location_input() -> dict:
    """
    Erlaubt Ad-hoc-Abfrage für benutzerdefinierten Standort
    """
    lat = float(input("📍 Breitengrad: "))
    lon = float(input("📍 Längengrad: "))
    name = input("🏙️ Ortsname: ")
    return {"lat": lat, "lon": lon, "name": name}

In [14]:
# Live-Daten abrufen (Forecast von Open-Meteo)
import requests
import pandas as pd
from datetime import datetime, timedelta


def clean_weather_data(df: pd.DataFrame) -> pd.DataFrame:
    """Bereinigt Live-Daten mit denselben Regeln wie die Feature-Pipeline."""
    df = df.copy()
    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    df["location_id"] = df.apply(
        lambda row: f"{row['latitude']}_{row['longitude']}", axis=1
    )
    df = df.dropna(subset=["time", "location_id"])
    df = df.drop_duplicates(subset=["location_id", "time"], keep="last")

    valid_ranges = {
        "temperature_2m": (-90, 60),
        "relative_humidity_2m": (0, 100),
        "precipitation": (0, 1000),
        "pressure_msl": (850, 1100),
        "surface_pressure": (850, 1100),
        "cloud_cover": (0, 100),
        "wind_speed_10m": (0, 250),
        "wind_gusts_10m": (0, 350),
        "cape": (0, 10000),
    }
    numeric_columns = list(valid_ranges)
    for column, (lower, upper) in valid_ranges.items():
        values = pd.to_numeric(df[column], errors="coerce")
        df[column] = values.mask((values < lower) | (values > upper))

    df = df.sort_values(["location_id", "time"])
    for column in numeric_columns:
        df[column] = df.groupby("location_id")[column].transform(
            lambda values: values.interpolate(limit_direction="both")
        )
        df[column] = df[column].fillna(
            df.groupby("location_id")[column].transform("median")
        )

    required_columns = [
        "temperature_2m", "relative_humidity_2m", "precipitation",
        "pressure_msl", "surface_pressure", "cloud_cover",
        "wind_speed_10m", "wind_gusts_10m", "cape",
    ]
    return df.dropna(subset=required_columns).reset_index(drop=True)


def fetch_live_forecast(lat: float, lon: float, location_name: str) -> pd.DataFrame:
    """Holt aktuelle Wetterdaten und einen 3-Tage-Forecast von Open-Meteo."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": [
            "temperature_2m", "relative_humidity_2m", "precipitation", "rain",
            "pressure_msl", "surface_pressure", "cloud_cover",
            "wind_speed_10m", "wind_gusts_10m", "wind_direction_10m", "cape"
        ],
        "past_days": 1,
        # Drei Forecast-Tage dienen nur dem Datenabruf; der Zielhorizont beträgt 3 Stunden.
        "forecast_days": 3,
        "timezone": "UTC"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()["hourly"]

    df = pd.DataFrame(data)
    df["time"] = pd.to_datetime(df["time"])
    df["location"] = location_name
    df["latitude"] = lat
    df["longitude"] = lon
    df = clean_weather_data(df)

    print(f"✅ {len(df)} bereinigte Live-Datenpunkte für {location_name} geladen")
    return df

# Definierte Standorte
LOCATIONS = {
    "Munich": (48.1351, 11.5820),
    "Hamburg": (53.5511, 9.9937)
}

In [15]:
# Modell aus der Model Registry herunterladen
import joblib
import os

mr = project.get_model_registry()
model_version = os.getenv("HOPSWORKS_MODEL_VERSION")
if model_version:
    model_meta = mr.get_model(
        name="severe_weather_classifier",
        version=int(model_version),
    )
else:
    available_models = mr.get_models(name="severe_weather_classifier")
    if not available_models:
        raise RuntimeError("Das Modell severe_weather_classifier wurde nicht gefunden.")
    model_meta = max(available_models, key=lambda candidate: int(candidate.version))

model_dir = model_meta.download()
print(f"✅ Modell heruntergeladen nach: {model_dir}")

model_path = os.path.join(model_dir, "model.joblib")
model = joblib.load(model_path)

print(f"✅ Modell geladen: {model_meta.name} (v{model_meta.version})")
print(f"📊 Trainings-Metriken: {model_meta.training_metrics}")

Downloading: 0.000%|          | 0/456659 elapsed<00:00 remaining<?

Downloading: 0.000%|          | 0/199 elapsed<00:00 remaining<?

✅ Modell heruntergeladen nach: /tmp/hopsworks/models/fhnw_p1_weather_forcasts/severe_weather_classifier/19/severe_weather_classifier_19
✅ Modell geladen: severe_weather_classifier (v19)
📊 Trainings-Metriken: {'n_test_samples': 373.0, 'n_train_samples': 1489.0, 'f1_score': 0.47058823529411764, 'roc_auc': 0.856728778467909, 'forecast_horizon_hours': 3.0, 'positive_class_ratio': 0.07387508394895903}


In [16]:
# Real-Time Single Prediction
# Drei-Stunden-Prognosehorizont für das trainierte Unwetterlabel.
FORECAST_HORIZON_HOURS = 3


def run_realtime_prediction(model, feature_vector: dict, threshold: float = 0.3) -> dict:
    """Führt eine Vorhersage für ein Sturmereignis in drei Stunden durch."""
    expected_features = getattr(model, "feature_names_in_", None)
    if expected_features is None:
        expected_features = model.get_booster().feature_names
    if expected_features is None or len(expected_features) == 0:
        raise ValueError("Das Modell enthält keine Feature-Namen für die Inference.")

    missing_features = [
        feature_name for feature_name in expected_features
        if feature_name not in feature_vector
    ]
    if missing_features:
        raise KeyError(f"Fehlende Modell-Features: {missing_features}")

    X = pd.DataFrame([
        {feature_name: feature_vector[feature_name] for feature_name in expected_features}
    ])
    X = X.apply(pd.to_numeric, errors="coerce")

    probability = model.predict_proba(X)[0, 1]
    warning = bool(probability >= threshold)

    result = {
        "storm_probability": round(float(probability), 4),
        "storm_warning": warning,
        "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
        "forecast_description": "Sturmrisiko in etwa 3 Stunden",
        "risk_level": "🔴 HOCH" if probability >= 0.6 else
                       "🟠 MITTEL" if probability >= threshold else "🟢 NIEDRIG"
    }
    return result

# Ausführen
result = run_realtime_prediction(model, live_vector, threshold=0.3)
print(f"⚡ Real-Time Ergebnis: {result}")

⚡ Real-Time Ergebnis: {'storm_probability': 0.0009, 'storm_warning': False, 'forecast_horizon_hours': 3, 'forecast_description': 'Sturmrisiko in etwa 3 Stunden', 'risk_level': '🟢 NIEDRIG'}
